# Import Environment variables

In [1]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

Found bucket: id=rw-migration-aou-rw-f7a4d148, bucketName=rw-migration-aou-rw-f7a4d148
-> Assigned migration variables (ID: rw-migration-aou-rw-f7a4d148)
Found bucket: id=temporary-workspace-bucket, bucketName=temporary-workspace-bucket-wb-perky-cabbage-8342
Found bucket: id=workspace-bucket, bucketName=workspace-bucket-wb-perky-cabbage-8342
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-perky-cabbage-8342
WORKSPACE_BUCKET: gs://workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_TEMP_BUCKET: gs://temporary-workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-f7a4d148
bucket_id_migrated: rw-migration-aou-rw-f7a4d148

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
Multi-trait-GWAS-in-admixed-populations GIT_REPO
dataset_test2 BQ_DATASET
prep_C2024Q3R9 BQ_DATASET
rw-mig

In [2]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

WORKSPACE_CDR = wb-silky-artichoke-2408.C2024Q3R9
WORKSPACE_BUCKET = gs://workspace-bucket-wb-perky-cabbage-8342
GOOGLE_PROJECT = wb-perky-cabbage-8342
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.R
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.sas


# Librairy

In [3]:
import os
import numpy as np
import pandas as pd
from google.cloud import bigquery

# Data's import

## Clinical data

In [4]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "df_54_RiskFactors_CurrentSmoking_AlcoholConsumption.tsv"

df = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

In [5]:
df.columns

Index(['person_id', 'inclusion_date', 'first_breast_cancer_date', 'delay_days',
       'age_at_inclusion', 'has_bc', 'eur_rye', 'eas_rye', 'amr_rye',
       'afr_rye', 'sas_rye', 'mid_rye', 'dominant_origin', 'ancestry_80',
       'biopsy_result', 'age_category', 'race_us_bcsc', 'race_us_bcsc_detail',
       'breast_cancer_family_history_first_degree', 'tobacco_survey_date',
       'tobacco_answer_concept_id', 'tobacco_answer_label', 'smoking_status',
       'date_releve', 'jours_semaine', 'verres_jour_conso',
       'lifetime_intake_of_alcohol', 'alcohol_intake_category'],
      dtype='object')

# Contextual informations

__Sept facteurs de risque de cancer du sein ont été pris en compte :__

* l’âge des premières règles ;
* la parité (antécédents d’accouchement) ;
* l’âge à la première grossesse à terme ;
* l’indice de masse corporelle (IMC) à l’âge adulte chez les femmes ménopausées ;
* la taille à l’âge adulte ;
* le recours actuel à un traitement hormonal de la ménopause (THM) à base d’œstrogènes et de progestérone ;
* la consommation moyenne d’alcool au cours de la vie.

__Harmonisation des données et définitions des variables__

* _Les variables dépendantes du temps ont été évaluées à une date de référence définie comme la date du diagnostic pour les cas et la date de l'entretien pour les témoins dans les études cas-témoins. [..] la date de référence était celle du dernier questionnaire de suivi, si disponible ; sinon, la date du questionnaire initial a été utilisée._

* _En l'absence de données sur le statut ménopausique, nous avons utilisé l'âge médian (54 ans) comme indicateur de substitution : les femmes âgées de moins de 54 ans ont été considérées comme préménopausiques et celles âgées de 54 ans ou plus comme postménopausiques._

* _Le recours actuel à un THM à base d’œstrogènes et de progestérone a été défini comme un recours dans les six mois précédant la date de référence._

* _Dans les études cas-témoins, l’IMC a été calculé à partir du poids habituel à l’âge adulte ou du poids un an avant la date de référence, si cette donnée était disponible. Si cette variable n’était pas disponible, le poids au début de l’âge adulte a été utilisé comme indicateur. Le poids déclaré au moment du diagnostic ou lors de l’entretien dans les études cas-témoins n’a pas été utilisé afin d’éviter l’influence de la maladie sur le poids. Pour les deux études de cohorte prospectives (MCCS, UKBGS), nous avons utilisé le poids déclaré lors de l'entretien initial (avant le diagnostic)._

* _Les variables continues (âge des premières règles, AFTP, consommation d'alcool, taille et IMC) ont été modélisées à la fois comme des variables continues et catégorielles_

Source : [Associations conjointes d'un score de risque polygénique et de facteurs de risque environnementaux pour le cancer du sein dans le Breast Cancer Association Consortium](https://pmc.ncbi.nlm.nih.gov/articles/PMC5913605/#sec16)

# Age at menopause

## Data import

In [6]:
# Connexion au CDR de ton workspace Verily
dataset = os.environ["WORKSPACE_CDR"]
client = bigquery.Client()

# Requête SQL combinant le standard OMOP (SNOMED) et le code source (ICD10CM)
query_menopause_exhaustive = f"""
WITH concepts_descendants AS (
    -- Concepts standards OMOP (SNOMED 4128329 + sa descendance)
    SELECT descendant_concept_id AS concept_id
    FROM `{dataset}.concept_ancestor`
    WHERE ancestor_concept_id = 4128329
),
concepts_icd10 AS (
    -- Concepts sources ICD10CM (Z78.0)
    SELECT concept_id
    FROM `{dataset}.concept`
    WHERE concept_code LIKE 'Z78.0%' 
      AND vocabulary_id = 'ICD10CM'
)
SELECT DISTINCT
    co.person_id,
    p.year_of_birth,
    co.condition_start_date AS date_at_menopause,
    
    -- Calcul de l'âge de la participante au moment du relevé
    EXTRACT(YEAR FROM co.condition_start_date) - p.year_of_birth AS age_at_menopause,
    
    -- Détails sur le codage (Standard vs Source)
    co.condition_concept_id AS omop_standard_id,
    c_std.concept_name AS omop_standard_libelle,
    co.condition_source_value AS code_icd10_brut,
    c_src.concept_name AS icd10_libelle

FROM `{dataset}.condition_occurrence` co
JOIN `{dataset}.person` p ON co.person_id = p.person_id
LEFT JOIN `{dataset}.concept` c_std ON co.condition_concept_id = c_std.concept_id
LEFT JOIN `{dataset}.concept` c_src ON co.condition_source_concept_id = c_src.concept_id

WHERE 
    -- 1. Filtre sur le concept standard et ses enfants
    co.condition_concept_id IN (SELECT concept_id FROM concepts_descendants)
    -- 2. Filtre sur le code source ICD10 (Z78.0)
    OR co.condition_source_concept_id IN (SELECT concept_id FROM concepts_icd10)
    OR co.condition_source_value LIKE 'Z78.0%'

ORDER BY co.person_id, co.condition_start_date
"""

# Exécution de la requête
df_menopause_all = client.query(query_menopause_exhaustive).to_dataframe()

# Affichage des résultats
print(f"Nombre total de relevés extraits : {len(df_menopause_all)}")
print(f"Nombre de patientes uniques identifiées : {df_menopause_all['person_id'].nunique()}")

pd.set_option('display.max_columns', None)
display(df_menopause_all.head(5))

Nombre total de relevés extraits : 68732
Nombre de patientes uniques identifiées : 27574


,person_id,year_of_birth,date_at_menopause,age_at_menopause,omop_standard_id,omop_standard_libelle,code_icd10_brut,icd10_libelle
0,1000091,1954,2019-12-27,65,4128329,Menopause present,Z78.0,Asymptomatic menopausal state
1,1000091,1954,2021-01-13,67,4128329,Menopause present,Z78.0,Asymptomatic menopausal state
2,1000095,1978,2020-05-06,42,4128329,Menopause present,Z78.0,Asymptomatic menopausal state
3,1000095,1978,2020-08-15,42,4128329,Menopause present,Z78.0,Asymptomatic menopausal state
4,1000127,1959,2017-12-06,58,4128329,Menopause present,Z78.0,Asymptomatic menopausal state


## Selecting the earliest event date

In [7]:
# Tri par date puis suppression des doublons en ne gardant que la première occurrence (la plus ancienne)
df_menopause_first_date = (
    df_menopause_all.sort_values(by=["person_id", "date_at_menopause"])
    .groupby("person_id")
    .first()
    .reset_index()
)

In [8]:
# Details of menopausal status
df_menopause_first_date.drop_duplicates()['omop_standard_libelle'].value_counts()

omop_standard_libelle
Menopause present            27440
Postmenopausal state           118
Asymptomatic                    15
Postmenopausal osteopenia        1
Name: count, dtype: int64

## Add column `menopausal_status`

In [9]:
df_menopause_first_date['menopausal_status'] = 1

## Median age of postmenopausal women

In [10]:
print(df_menopause_first_date['age_at_menopause'].median())

66.0


In [11]:
df_menopause_my_indivs = df_menopause_first_date[df_menopause_first_date['person_id'].isin(list(df['person_id']))]

df_menopause_my_indivs

,person_id,year_of_birth,date_at_menopause,age_at_menopause,omop_standard_id,omop_standard_libelle,code_icd10_brut,icd10_libelle,menopausal_status
5,1000402,1951,2016-07-14,65,4128329,Menopause present,Z78.0,Asymptomatic menopausal state,1
6,1000434,1956,2022-12-07,66,4128329,Menopause present,Z78.0,Asymptomatic menopausal state,1
7,1000678,1952,2018-03-09,66,4128329,Menopause present,289903006,Menopause present,1
8,1000803,1965,2019-01-25,54,4128329,Menopause present,Z78.0,Asymptomatic menopausal state,1
10,1000921,1952,2018-09-01,66,4128329,Menopause present,Z78.0,Asymptomatic menopausal state,1
...,...,...,...,...,...,...,...,...,...
27565,9989434,1949,2016-12-13,67,4128329,Menopause present,Z78.0,Asymptomatic menopausal state,1
27568,9990917,1966,2022-01-27,56,4128329,Menopause present,Z78.0,Asymptomatic menopausal state,1
27569,9992398,1963,2023-03-29,60,4128329,Menopause present,Z78.0,Asymptomatic menopausal state,1
27571,9993246,1948,2021-02-09,73,4128329,Menopause present,Z78.0,Asymptomatic menopausal state,1


# Body mass index (BMI) [Percentile]

## Data import

### Description méthodologique de l'extraction (IMC direct)

__Périmètre et Source des Données :__
_L'extraction vise à récupérer l'ensemble des mesures d'indice de masse corporelle (IMC / Body Mass Index) pré-calculées et enregistrées dans les dossiers médicaux électroniques (EHR) des participantes de la cohorte All of Us._

__Stratégie de Cartographie des Concepts (Mapping) :__
Afin de garantir l'exhaustivité de la collecte et de pallier l'hétérogénéité des codages hospitaliers à la source, la requête croise trois niveaux de terminologies au sein du modèle OMOP CDM :
1. __La hiérarchie des concepts standards OMOP :__ Utilisation de la table `concept_ancestor` pour capturer le concept d'IMC adulte principal (`concept_id = 3038553`) ainsi que l'ensemble de ses concepts descendants.
2. __Le vocabulaire LOINC :__ Ingestion directe du code LOINC `39156-5` (Body mass index (BMI) [Ratio]).
3. __Les valeurs sources brutes :__ Filtrage sur la colonne `measurement_source_value` (via la clause `LIKE '39156-5%'`) pour récupérer les enregistrements hospitaliers dont le code source n'aurait pas été traduit lors du processus d'ETL.

__Contrôle Qualité et Invariants Cliniques :__
Un filtre de cohérence biologique est appliqué à la source (`value_as_number BETWEEN 10 AND 75`) pour éliminer les erreurs de saisie administrative, les valeurs nulles et les aberrations extrêmes (ex. valeurs hors plage physiologique ou percentiles mal étiquetés).Structure des Données Extraites :Le jeu de données résultant est structuré sous forme longitudinale (plusieurs lignes horodatées par participante), conservant la date exacte de l'événement (`date_evenement`) et l'âge de la participante au moment de la mesure (`age_au_releve`). Cette structure permet l'alignement ultérieur des mesures sur la fenêtre temporelle d'intérêt ($T_{\text{réf}} - 1\text{ an}$).

In [12]:
dataset = os.environ["WORKSPACE_CDR"]
client = bigquery.Client()

query_bmi_exhaustif = f"""
WITH all_bmi_concepts AS (
    -- Concept standard IMC adulte (LOINC 39156-5) + sa descendance
    SELECT descendant_concept_id AS concept_id
    FROM `{dataset}.concept_ancestor`
    WHERE ancestor_concept_id = 3038553
    
    UNION DISTINCT
    
    -- Ingestion directe du concept LOINC 39156-5
    SELECT concept_id
    FROM `{dataset}.concept`
    WHERE concept_code = '39156-5' AND vocabulary_id = 'LOINC'
)

SELECT 
    m.person_id,
    p.year_of_birth,
    m.measurement_date AS date_evenement,
    EXTRACT(YEAR FROM m.measurement_date) - p.year_of_birth AS age_au_releve,
    m.value_as_number AS bmi_valeur,
    c_std.concept_name AS nom_concept,
    m.measurement_source_value AS code_source_brut
FROM `{dataset}.measurement` m
JOIN `{dataset}.person` p ON m.person_id = p.person_id
LEFT JOIN `{dataset}.concept` c_std ON m.measurement_concept_id = c_std.concept_id
WHERE (
    m.measurement_concept_id IN (SELECT concept_id FROM all_bmi_concepts)
    OR m.measurement_source_value LIKE '39156-5%'
)
AND m.value_as_number BETWEEN 10 AND 75 -- Filtre clinique sur les valeurs d'IMC valides
ORDER BY m.person_id, m.measurement_date ASC
"""

df_bmi_all = client.query(query_bmi_exhaustif).to_dataframe()

print(f"Nombre total de relevés d'IMC trouvés : {len(df_bmi_all)}")
print(f"Nombre de femmes uniques avec au moins un IMC : {df_bmi_all['person_id'].nunique()}")

Nombre total de relevés d'IMC trouvés : 8536156
Nombre de femmes uniques avec au moins un IMC : 505883


In [14]:
df_bmi_my_indivs = df_bmi_all[df_bmi_all['person_id'].isin(list(df['person_id']))]

### Description méthodologique de l'extraction (IMC dérivé des anthropométries brutes)

__Périmètre et Source des Données :__
Cette méthode vise à calculer l'indice de masse corporelle (IMC / Body Mass Index) à partir des mesures anthropométriques primaires (taille et poids) consignées soit lors des consultations hospitalières (EHR), soit lors de l'évaluation physique initiale à l'inclusion dans le programme All of Us (Physical Measurements / PM).

__Stratégie de Collecte et d'Harmonisation :__
L'extraction s'appuie sur la table `measurement` du modèle OMOP CDM en isolant simultanément deux constantes biologiques :
1. __La Taille :__ Identifiée via le code LOINC `8302-2` (concepts OMOP `3036277` et `3023540`). Une règle de conversion automatique est appliquée pour harmoniser les unités en mètres (`value_as_number / 100.0` lorsque la mesure est saisie en centimètres).
2. __Le Poids :__ Identifié via le code LOINC `29463-7` (concepts OMOP `3025315` et `3013762`), exprimé en kilogrammes.

__Appariement Temporel et Calcul de l'IMC :__
Une jointure stricte (`INNER JOIN`) est réalisée entre les relevés de poids et de taille en exigeant une concordance exacte sur l'identifiant du participant (`person_id`) et la date de mesure (`date_evenement`). L'IMC brut est ensuite calculé à la date de l'examen selon la formule standard :$$\text{IMC} = \frac{\text{Poids (kg)}}{\text{Taille (m)}^2}$$

__Contrôle Qualité et Invariants Cliniques :__
Afin d'écarter les erreurs de saisie et les artefacts de numérisation, des bornes physiologiques strictes sont appliquées :
* Poids compris entre __30 kg et 250 kg__ ;
* Taille comprise entre __1,20 m et 2,20 m__ ;
* IMC dérivé compris dans la plage clinique valide de 10 __kg/m² à 75 kg/m²__.

__Structure et Traçabilité :__
La table résultante conserve le type de provenance (type_provenance) afin d'identifier l'origine de la donnée (EHR vs visite d'inclusion PM), ainsi que l'âge exact lors de la mesure (age_au_releve). Cette approche garantit une excellente complétude et fournit une série temporelle robuste pour l'alignement ultérieur sur la date de référence ($T_{\text{réf}} - 1\text{ an}$).

In [15]:
dataset = os.environ["WORKSPACE_CDR"]
client = bigquery.Client()

query_height_weight = f"""
WITH height_data AS (
    -- Extraction et conversion de la taille en cm entiers (arrondis sans virgule)
    SELECT 
        m.person_id,
        m.measurement_date AS date_evenement,
        m.measurement_type_concept_id,
        c_type.concept_name AS type_provenance,
        CASE 
            WHEN m.value_as_number <= 3 THEN ROUND(m.value_as_number * 100.0) -- Conversion m -> cm
            ELSE ROUND(m.value_as_number) -- Déjà en cm
        END AS height_cm
    FROM `{dataset}.measurement` m
    LEFT JOIN `{dataset}.concept` c_type ON m.measurement_type_concept_id = c_type.concept_id
    WHERE m.measurement_concept_id IN (3036277, 3023540) -- Taille (LOINC 8302-2)
       OR m.measurement_source_value LIKE '8302-2%'
),

weight_data AS (
    -- Extraction du poids (en kg) arrondi à 1 décimale
    SELECT 
        m.person_id,
        m.measurement_date AS date_evenement,
        m.measurement_type_concept_id,
        c_type.concept_name AS type_provenance,
        ROUND(m.value_as_number, 1) AS weight_kg
    FROM `{dataset}.measurement` m
    LEFT JOIN `{dataset}.concept` c_type ON m.measurement_type_concept_id = c_type.concept_id
    WHERE m.measurement_concept_id IN (3025315, 3013762) -- Poids (LOINC 29463-7)
       OR m.measurement_source_value LIKE '29463-7%'
)

-- Jointure de la taille et du poids à la MÊME date pour chaque participante
SELECT 
    w.person_id,
    p.year_of_birth,
    w.date_evenement,
    EXTRACT(YEAR FROM w.date_evenement) - p.year_of_birth AS age_au_releve,
    w.type_provenance,
    w.weight_kg,
    CAST(h.height_cm AS INT64) AS height_cm, -- Entier strict (sans virgule)
    -- Calcul de l'IMC brut (kg/m²) arrondi à 1 décimale
    ROUND(w.weight_kg / POWER(h.height_cm / 100.0, 2), 1) AS bmi_calcule
FROM weight_data w
JOIN height_data h 
  ON w.person_id = h.person_id 
 AND w.date_evenement = h.date_evenement
JOIN `{dataset}.person` p 
  ON w.person_id = p.person_id

WHERE w.weight_kg BETWEEN 30 AND 250 -- Nettoyage des valeurs aberrantes de poids
  AND h.height_cm BETWEEN 120 AND 220 -- Nettoyage des valeurs aberrantes de taille (en cm)
  AND (w.weight_kg / POWER(h.height_cm / 100.0, 2)) BETWEEN 10 AND 75 -- IMC valide

ORDER BY w.person_id, w.date_evenement ASC
"""

df_bmi_computed = client.query(query_height_weight).to_dataframe()

print(f"Nombre de relevés d'IMC calculés (Poids/Taille) : {len(df_bmi_computed)}")
print(f"Nombre de femmes uniques : {df_bmi_computed['person_id'].nunique()}")

Nombre de relevés d'IMC calculés (Poids/Taille) : 13304643
Nombre de femmes uniques : 510736


In [17]:
df_bmi_computed_my_indivs = df_bmi_computed[df_bmi_computed['person_id'].isin(list(df['person_id']))]

## Fusion des jeux de données d'IMC

### Description synthétique de la fusion des jeux de données d'IMC

Ce script réalise la consolidation du jeu de données d'IMC en combinant deux sources :

* __Complémentarité des sources :__ Il prend comme base `df_bmi_computed` (IMC dérivé des mesures brutes de poids et taille) et y ajoute uniquement les participantes absentes mais présentes dans `df_bmi_all` (IMC direct issu de l'EHR).
* __Harmonisation et concaténation :__ Les structures de colonnes sont alignées en attribuant la provenance (`type_provenance = 'EHR Direct BMI'`) et des valeurs nuls pour le poids/taille manquants du second jeu de données.
* __Organisation finale :__ Le jeu de données réuni (`df_bmi_combined`) est trié par participante et par ordre chronologique pour conserver l'historique complet des mesures.

In [18]:
import numpy as np
import pandas as pd

# 1. Identification des individus uniques dans chaque dataset
ids_computed = set(df_bmi_computed_my_indivs["person_id"])

# 2. Sélection des lignes uniques à df_bmi_my_indivs
df_bmi_all_missing = df_bmi_my_indivs[
    ~df_bmi_my_indivs["person_id"].isin(ids_computed)
].copy()

# 3. Préparation et typage avec les types annulables Pandas (Int64 / Float64)
df_bmi_all_missing = df_bmi_all_missing.rename(
    columns={"bmi_valeur": "bmi_calcule"}
)
df_bmi_all_missing["type_provenance"] = "EHR Direct BMI"

# Conversion explicite vers des dtypes compatibles avec NaN
df_bmi_all_missing["weight_kg"] = pd.Series(dtype="Float64")
df_bmi_all_missing["height_cm"] = pd.Series(dtype="Int64")

# Harmonisation des dtypes sur le premier DataFrame également
df_bmi_computed_temp = df_bmi_computed_my_indivs.copy()
df_bmi_computed_temp["height_cm"] = df_bmi_computed_temp["height_cm"].astype("Int64")
df_bmi_computed_temp["weight_kg"] = df_bmi_computed_temp["weight_kg"].astype("Float64")

# 4. Concaténation propre
cols_target = [
    "person_id",
    "year_of_birth",
    "date_evenement",
    "age_au_releve",
    "type_provenance",
    "weight_kg",
    "height_cm",
    "bmi_calcule",
]

df_bmi = pd.concat(
    [df_bmi_computed_temp[cols_target], df_bmi_all_missing[cols_target]],
    ignore_index=True,
)

# 5. Tri chronologique
df_bmi = df_bmi.sort_values(
    by=["person_id", "date_evenement"]
).reset_index(drop=True)

print(
    f"Nombre total de femmes uniques obtenues : {df_bmi['person_id'].nunique()}"
)

Nombre total de femmes uniques obtenues : 122100


In [19]:
df_bmi

,person_id,year_of_birth,date_evenement,age_au_releve,type_provenance,weight_kg,height_cm,bmi_calcule
0,1000045,1958,2011-10-25,53,No matching concept,63.3,157,25.7
1,1000045,1958,2019-06-27,61,No matching concept,60.3,157,24.5
2,1000045,1958,2019-08-19,61,No matching concept,60.3,157,24.5
3,1000045,1958,2020-02-27,62,From physical examination,60.0,158,24.0
4,1000045,1958,2021-01-19,63,No matching concept,59.9,157,24.3
...,...,...,...,...,...,...,...,...
3416784,9999678,1964,2020-10-29,56,From physical examination,85.9,168,30.4
3416785,9999678,1964,2023-04-13,59,From physical examination,82.2,165,30.2
3416786,9999678,1964,2023-04-13,59,From physical examination,82.0,165,30.1
3416787,9999678,1964,2023-04-17,59,From physical examination,81.2,165,29.8


## Add columns `adult_body_height` & `adult_BMI`

### `adult_body_height`

In [20]:
# Catégorisation de l'IMC (Adult BMI)
bmi_bins = [-np.inf, 18.5, 25.0, 30.0, np.inf]
bmi_labels = [
    "< 18.5 kg/m²",
    ">= 18.5 to < 25 kg/m²",
    ">= 25 to < 30 kg/m²",
    ">= 30 kg/m²",
]

df_bmi["bmi_category"] = pd.cut(
    df_bmi["bmi_calcule"], bins=bmi_bins, labels=bmi_labels, right=False
)

# Vérification des résultats
print("--- Répartition IMC ---")
print(df_bmi["bmi_category"].value_counts(sort=False))

--- Répartition IMC ---
bmi_category
< 18.5 kg/m²               39750
>= 18.5 to < 25 kg/m²     649460
>= 25 to < 30 kg/m²       810910
>= 30 kg/m²              1916669
Name: count, dtype: int64


### `adult_BMI`

In [21]:
# Catégorisation de la taille (Adult body height)
height_bins = [-np.inf, 158.0, 162.0, 165.0, 168.0, np.inf]
height_labels = [
    "< 158 cm",
    ">= 158 to < 162 cm",
    ">= 162 to < 165 cm",
    ">= 165 to < 168 cm",
    ">= 168 cm",
]

df_bmi["height_category"] = pd.cut(
    df_bmi["height_cm"],
    bins=height_bins,
    labels=height_labels,
    right=False,  # Exclut la borne supérieure pour respecter le '<'
)

print("\n--- Répartition Taille ---")
print(df_bmi["height_category"].value_counts(sort=False))


--- Répartition Taille ---
height_category
< 158 cm               850863
>= 158 to < 162 cm     512160
>= 162 to < 165 cm     536284
>= 165 to < 168 cm     455050
>= 168 cm             1060821
Name: count, dtype: int64


In [22]:
df_bmi

,person_id,year_of_birth,date_evenement,age_au_releve,type_provenance,weight_kg,height_cm,bmi_calcule,bmi_category,height_category
0,1000045,1958,2011-10-25,53,No matching concept,63.3,157,25.7,>= 25 to < 30 kg/m²,< 158 cm
1,1000045,1958,2019-06-27,61,No matching concept,60.3,157,24.5,>= 18.5 to < 25 kg/m²,< 158 cm
2,1000045,1958,2019-08-19,61,No matching concept,60.3,157,24.5,>= 18.5 to < 25 kg/m²,< 158 cm
3,1000045,1958,2020-02-27,62,From physical examination,60.0,158,24.0,>= 18.5 to < 25 kg/m²,>= 158 to < 162 cm
4,1000045,1958,2021-01-19,63,No matching concept,59.9,157,24.3,>= 18.5 to < 25 kg/m²,< 158 cm
...,...,...,...,...,...,...,...,...,...,...
3416784,9999678,1964,2020-10-29,56,From physical examination,85.9,168,30.4,>= 30 kg/m²,>= 168 cm
3416785,9999678,1964,2023-04-13,59,From physical examination,82.2,165,30.2,>= 30 kg/m²,>= 165 to < 168 cm
3416786,9999678,1964,2023-04-13,59,From physical examination,82.0,165,30.1,>= 30 kg/m²,>= 165 to < 168 cm
3416787,9999678,1964,2023-04-17,59,From physical examination,81.2,165,29.8,>= 25 to < 30 kg/m²,>= 165 to < 168 cm


## Date de référence pour l'IMC et le poids

__Description synthétique du traitement d'alignement temporel de l'IMC__

1. __Définition de la date de référence ($T_{\text{réf}}$) :__ Le script assigne une date pivot à chaque participante : la date de diagnostic pour les cas (`first_breast_cancer_date`) et la date d'inclusion dans l'étude pour les témoins (`inclusion_date`).
2. __Filtrage temporel strict selon la littérature :__ Pour éliminer le risque qu'une perte de poids liée à la maladie ou à ses traitements ne biaise la mesure, il fixe une date limite d'éligibilité :
* __Cas :__ Mesures effectuées au moins __1 an avant le diagnostic__ ($T_{\text{réf}} - 365\text{ jours}$).
* __Témoins :__ Mesures effectuées au plus tard à la date d'inclusion ($T_{\text{réf}}$).

3. __Sélection et fusion :__ Parmi l'ensemble des mesures valides antérieures à cette date limite, le script retient la plus récente pour chaque participante, puis réalise un Left Join pour rattacher l'IMC, le poids et la taille de référence à la cohorte globale.

In [25]:
import numpy as np
import pandas as pd

# 1. Formatage explicite des dates
df["inclusion_date"] = pd.to_datetime(df["inclusion_date"])
df["first_breast_cancer_date"] = pd.to_datetime(df["first_breast_cancer_date"])
df_bmi["date_evenement"] = pd.to_datetime(df_bmi["date_evenement"])

# 2. Définition explicite de T_ref :
# Date de cancer si elle existe (Cas), sinon date d'inclusion (Témoins)
df["date_reference"] = df["first_breast_cancer_date"].fillna(
    df["inclusion_date"]
)

# 3. Jointure avec la table d'IMC
df_merged = pd.merge(
    df[["person_id", "has_bc", "date_reference"]],
    df_bmi,
    on="person_id",
    how="inner",
)

# 4. Définition de la date limite selon le statut :
# Cas (1) : T_ref - 365 jours
# Témoins (0) : T_ref
df_merged["date_limite"] = np.where(
    df_merged["has_bc"] == 1,
    df_merged["date_reference"] - pd.Timedelta(days=365),
    df_merged["date_reference"],
)

# 5. Filtre : mesures antérieures ou égales à la date limite
df_eligible = df_merged[
    df_merged["date_evenement"] <= df_merged["date_limite"]
].copy()

# 6. Sélection de la mesure la plus récente avant la date limite
df_bmi_selected = (
    df_eligible.sort_values(
        by=["person_id", "date_evenement"], ascending=[True, False]
    )
    .groupby("person_id")
    .first()
    .reset_index()
)

# 7. Sélection des colonnes (incluant les nouvelles catégories) et renommage en anglais
cols_to_keep = [
    "person_id",
    "date_evenement",
    "weight_kg",
    "height_cm",
    "bmi_calcule",
    "bmi_category",
    "height_category",
]

df_bmi_selected = df_bmi_selected[cols_to_keep].rename(
    columns={
        "date_evenement": "bmi_measurement_date",
        "weight_kg": "ref_weight_kg",
        "height_cm": "ref_height_cm",
        "bmi_calcule": "ref_bmi",
        "bmi_category": "ref_bmi_category",
        "height_category": "ref_height_category",
    }
)

# 8. Fusion finale avec la cohorte principale (Left Join)
df_final = pd.merge(df, df_bmi_selected, on="person_id", how="left")

# Contrôle des effectifs
print(
    f"Controls with valid BMI: {df_final[df_final['has_bc'] == 0]['ref_bmi'].notna().sum()}"
)
print(
    f"Cases with valid BMI (T_ref - 1 yr): {df_final[df_final['has_bc'] == 1]['ref_bmi'].notna().sum()}"
)

df_final.head()

Controls with valid BMI: 92940
Cases with valid BMI (T_ref - 1 yr): 707


,person_id,inclusion_date,first_breast_cancer_date,delay_days,age_at_inclusion,has_bc,eur_rye,eas_rye,amr_rye,afr_rye,sas_rye,mid_rye,dominant_origin,ancestry_80,biopsy_result,age_category,race_us_bcsc,race_us_bcsc_detail,breast_cancer_family_history_first_degree,tobacco_survey_date,tobacco_answer_concept_id,tobacco_answer_label,smoking_status,date_releve,jours_semaine,verres_jour_conso,lifetime_intake_of_alcohol,alcohol_intake_category,date_reference,bmi_measurement_date,ref_weight_kg,ref_height_cm,ref_bmi,ref_bmi_category,ref_height_category
0,1700611,2019-09-17,NaT,NaN,70.255989,0,0.091868,0.000000,0.000000,0.848348,0.000000,0.059785,afr_rye,afr_rye,0,72,2,afr_rye,0,2019-09-17,1585858.0,100 Cigs Lifetime: Yes,1.0,2019-09-17,0.00,0.0,0.0,< 0.00001 g/day,2019-09-17,2019-09-17,67.1,178,21.2,>= 18.5 to < 25 kg/m²,>= 168 cm
1,1356439,2019-03-04,NaT,NaN,69.716632,0,0.065172,0.000000,0.025094,0.000000,0.809500,0.100234,sas_rye,sas_rye,0,67,3,eas_rye/sas_rye,0,2019-03-04,1585859.0,100 Cigs Lifetime: No,0.0,NaN,NaN,NaN,NaN,NaN,2019-03-04,NaT,<NA>,<NA>,NaN,NaN,NaN
2,3451057,2023-01-18,NaT,NaN,71.594798,0,0.103268,0.000000,0.037591,0.000000,0.847168,0.011974,sas_rye,sas_rye,0,72,3,eas_rye/sas_rye,0,2023-01-18,1585859.0,100 Cigs Lifetime: No,0.0,NaN,NaN,NaN,NaN,NaN,2023-01-18,2023-01-11,48.1,157,19.5,>= 18.5 to < 25 kg/m²,< 158 cm
3,1559161,2019-03-11,NaT,NaN,67.737166,0,0.006018,0.006429,0.965105,0.000000,0.022449,0.000000,amr_rye,amr_rye,0,67,8,amr_rye/mid_rye,0,2019-03-14,1585859.0,100 Cigs Lifetime: No,0.0,2019-03-14,0.25,1.5,0.8,>= 0.00001 to < 5 g/day,2019-03-11,NaT,<NA>,<NA>,NaN,NaN,NaN
4,1468526,2019-10-07,NaT,NaN,63.310062,0,0.896282,0.000000,0.032661,0.000000,0.071057,0.000000,eur_rye,eur_rye,0,62,1,eur_rye,0,2019-10-07,1585858.0,100 Cigs Lifetime: Yes,1.0,NaN,NaN,NaN,NaN,NaN,2019-10-07,2019-10-07,72.9,162,27.8,>= 25 to < 30 kg/m²,>= 162 to < 165 cm


In [30]:
df_final[["has_bc","ref_bmi_category"]].value_counts(sort=False)

has_bc  ref_bmi_category     
0       < 18.5 kg/m²              1171
        >= 18.5 to < 25 kg/m²    21305
        >= 25 to < 30 kg/m²      24970
        >= 30 kg/m²              45494
1       < 18.5 kg/m²                 8
        >= 18.5 to < 25 kg/m²      166
        >= 25 to < 30 kg/m²        194
        >= 30 kg/m²                339
Name: count, dtype: int64

In [31]:
df_final[["has_bc","ref_height_category"]].value_counts(sort=False)

has_bc  ref_height_category
0       < 158 cm               22456
        >= 158 to < 162 cm     18650
        >= 162 to < 165 cm     14672
        >= 165 to < 168 cm     12354
        >= 168 cm              24482
1       < 158 cm                 188
        >= 158 to < 162 cm       123
        >= 162 to < 165 cm       113
        >= 165 to < 168 cm        80
        >= 168 cm                203
Name: count, dtype: int64

# Data's export

In [32]:
destination_filename = 'Datas/df_54_RiskFactors_CurrentSmoking_AlcoholConsumption_Menopause_BMI_Height.tsv'
df_final.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./Datas/df_54_RiskFactors_CurrentSmoking_AlcoholConsumption_Menopause_BMI_Height.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/ 34.6 MiB]                                                \r/ [1 files][ 34.6 MiB/ 34.6 MiB]                                                \r-\r\nOperation completed over 1 objects/34.6 MiB.                                     \n'